### Практическое задание
К нам в аналитический отдел пришел менеджер Сильвестр Андреевич, занимающийся премиальными услугами. У него только-только завершилась встреча с членами правления, по итогу которой успехи отдела продаж были поставлены под сомнение. Бизнес оперирует гипотезой, что премиальные услуги продаются хуже, чем у конкурентов, и не догоняют ожидаемый тренд, потому что отдел маркетинга принимает неудачные решения. Сильвестр Андреевич берет дело под свой контроль и хочет мониторить работу отдела продаж, сравнивая продажи в первый месяц с прогнозируемыми значениями.

Наша миссия состоит в том, чтобы сконструировать модель, прогнозирующую сумму выручки с премиальных услуг у клиентов в первый месяц их работы с сервисом. Работать это должно следующим образом:

сервисом начал пользоваться новый клиент, завел профиль,
мы обработали информацию из профиля, создали фичи и дали прогноз.
Обучать модель будем, как всегда, на прецедентах. Будем наблюдать историю оплат у клиентов в первый месяц и пытаться ее описать алгоритмом линейной регрессии.

Заодно потренируемся писать свои селекторы и трансформеры, использовать пайплайны, запускать кросс-валидацию и подбирать гиперпараметры.

Вам снова предстоит работать с данными классифайда. Но в этот раз нет необходимости подключаться к БД: мы будем использовать отдельный файл с данными: `premium_by_passports.csv`

Они, конечно же, взяты из той же БД путем джоина некоторых таблиц, просто содержат в себе пропуски, чтобы мы могли потренироваться с обработкой данных.

Схема данных:
- `payment_date` - дата транзакции
- `type` - тип операции (премиальные услуги)
- `passport_id` - идентификатор профиля клиента (ключ для определения клиента)
- `created_at` - дата создания профиля клиента
- `user_type_name` - тип пользователя
- `user_type_cars_name` - тип пользователя в машинах
- `revenue` - сумма операции, выручка с нее

In [1]:
import pandas as pd
from datetime import timedelta
from datetime import datetime as dt
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV


import warnings

warnings.filterwarnings("ignore")

### Задача 1. Собираем датасеты (1/5)
В этой задаче нас интересует сумма выручки с премиальных услуг у клиентов в первый месяц их работы с сервисом. Так что в данных можно было бы по каждому клиенту оставить только его платежи, совершённые в первый месяц (и скоро мы именно так и сделаем).

Но дополнительной важной информацией может быть общая сумма платежей (выручка) по каждому месяцу. Например, мы можем использовать эту информацию при создании признаков.

В реальной жизни такие данные могли бы приходить отдельно, из другого источника (в котором информация автоматически обновляется раз в месяц).

Но в этом задании мы подготовим такие данные самостоятельно на основе того же файла с данными (и притворимся, что это "внешние данные").

Посчитайте сумму выручки по каждому месяцу каждого года.

Создайте из полученных данных словарь `quasi_external` такого вида, как в примере ниже. Он пригодится нам позже.

```python
{
    '2021-02': сумма выручки,
    '2021-03': сумма выручки,
    '2021-04': сумма выручки,
    '2021-05': сумма выручки
    ...
}
```
Какая сумма транзакций получилась за июнь 2022 года?

In [2]:
data = pd.read_csv("premium_by_passports.csv")

In [3]:
data.head()

,payment_date,type,passport_id,created_at,user_type_name,user_type_cars_name,revenue
0,2021-04-20,premium,140980663,2021-03-29 23:33:14,profi,cars_seller,1370
1,2022-11-07,premium,141788719,2021-07-16 14:25:39,simple_user,NaN,785
2,2022-11-29,premium,140458955,2021-01-16 00:12:07,simple_user,cars_simple,985
3,2022-07-03,premium,143665334,2022-05-31 21:26:31,simple_user,NaN,785
4,2022-11-02,premium,143267208,2022-03-20 21:17:48,simple_user,cars_simple,1105


In [4]:
data["payment_date"] = pd.to_datetime(data["payment_date"])

quasi_external = (
    data.groupby(data["payment_date"].dt.strftime("%Y-%m"))
    .agg(revenue=("revenue", "sum"))
    .to_dict()
)["revenue"]

In [5]:
quasi_external["2022-06"]

6537800

### Задача 1. Собираем датасеты (2/5)
Посмотрите на данные, загруженные из файла `premium_by_passports.csv`.

Видите ли вы какие-то противоречия, нестыковки, странности в данных?



In [6]:
data.head()

,payment_date,type,passport_id,created_at,user_type_name,user_type_cars_name,revenue
0,2021-04-20,premium,140980663,2021-03-29 23:33:14,profi,cars_seller,1370
1,2022-11-07,premium,141788719,2021-07-16 14:25:39,simple_user,NaN,785
2,2022-11-29,premium,140458955,2021-01-16 00:12:07,simple_user,cars_simple,985
3,2022-07-03,premium,143665334,2022-05-31 21:26:31,simple_user,NaN,785
4,2022-11-02,premium,143267208,2022-03-20 21:17:48,simple_user,cars_simple,1105


In [7]:
passport_df = data.groupby("passport_id", as_index=False).agg(
    payment_date=("payment_date", "min"), created_at=("created_at", "min")
)
passport_df.head()

,passport_id,payment_date,created_at
0,140347347,2022-07-21,2021-01-01 01:33:37
1,140347365,2022-12-26,2021-01-01 01:39:22
2,140347525,2021-04-05,2021-01-01 03:24:09
3,140347855,2021-02-10,2021-01-01 10:58:28
4,140348223,2022-07-12,2021-01-01 12:57:14


In [8]:
passport_df["created_at"] = pd.to_datetime(
    passport_df["created_at"]
).dt.strftime("%Y-%m-%d")

passport_df.query("payment_date < created_at")

,passport_id,payment_date,created_at
7548,141402613,2022-08-18,2023-01-31


### Задача 1. Собираем датасеты (3/5)
Можно заметить, что у одного из клиентов все платежи совершены раньше даты создания профиля. Причём платежи совершены в 2022 году, а профиль создан в 2023.

Поскольку мы не знаем настоящую дату создания профиля, мы не сможем и определить, какие из платежей сделаны в первый месяц. Так что просто удалим данные этого клиента из датафрейма.

Введите `passport_id` и количество удалённых вами строк (количество платежей, которое было у этого клиента). Разделите значения запятой и пробелом (например, 2128506, 27).

In [9]:
wrong_passport_id = passport_df.query("payment_date < created_at")[
    "passport_id"
].iloc[0]

In [10]:
new_data = data[data["passport_id"] != wrong_passport_id]

In [11]:
print(str(wrong_passport_id) + ",", data.shape[0] - new_data.shape[0])

141402613, 41


### Задача 1. Собираем датасеты (4/5)
Столбцы `type`, `created_at`, `user_type_name`, `user_type_cars_name`, по сути, являются свойствами клиента, а не платежа. Они должны быть одинаковыми для всех строк в пределах одного `passport_id`. Так ли это у нас?

Видим что прямых противоречий в данных в этом отношении нет. Не бывает такого, что у одного `passport_id` две разные даты `created_at`. Или что один и тот же клиент является то `simple_user`, то `profi` (проверьте самостоятельно, что всё так).

Но для некоторых клиентов данные неполные.

Посмотрите, например, платежи клиента с `passport_id = 140357053`. Обратите внимание на столбцы `user_type_name` и `user_type_cars_name`. Где-то значения есть, где-то пропущены. Может быть и такое, что у клиента два платежа, в одном указан только `user_type_name`, а в другом только `user_type_cars_name` (посмотрите, например, `passport_id` `140377309`). Мы можем "собрать" значения из разных строк, но нет строки, где одновременно не пусты оба столбца.

Соберём датафрейм, где для каждого `passport_id` будут значения `user_type_name` и `user_type_cars_name`, которые мы можем "по максимуму" вытащить из датафрейма.

Лучше сделать это сейчас, на всех доступных платежах, до того, как мы по каждому клиенту оставим только платежи за первый месяц. Иначе часть информации может потеряться.

Взять первое непустое значение столбца по каждому `passport_id` могло бы быть не очень тривиальной задачей. Но, к счастью, `first` в pandas работает именно так: берёт первое непустое значение в каждой группе.

Итак, сгруппируйте текущий датафрейм по `passport_id`. При агрегации используйте `first` для двух столбцов: `user_type_name` и `user_type_cars_name` (с остальными столбцами поработаем позже). При группировке можете использовать параметр `as_index=False`, чтобы результат был в более удобном формате. Проверьте, что получилось.

Несмотря на все усилия, не для всех клиентов удалось восстановить значения этих двух столбцов.

Через запятую и пробел введите количество строк в полученном датафрейме, долю пропусков в `user_type_name` и долю пропусков в `user_type_cars_name`. Доли округлите до 2 знаков после точки. Пример: `10500, 0.07, 0.21`.



In [12]:
passport_id_df = new_data.groupby("passport_id", as_index=False).agg(
    user_type_name=("user_type_name", "first"),
    user_type_cars_name=("user_type_cars_name", "first"),
)

In [13]:
all_rows = passport_id_df.shape[0]

print(
    all_rows,
    round(passport_id_df["user_type_name"].isna().sum() / all_rows, 2),
    round(passport_id_df["user_type_cars_name"].isna().sum() / all_rows, 2),
    sep=", ",
)

28604, 0.03, 0.42


### Задача 1. Собираем датасеты (5/5)
Соберём основной датасет, с которым будем работать в этом задании. По каждому `passport_id` нас интересуют только платежи за первый месяц, так что возьмём только их.

Более конкретно, берём только платежи, которые были совершены клиентом в пределах 30 дней после онбординга. Т.е. такие, где разность между `payment_date` и датой `created_at` меньше или равна 30 дней.

Обратите внимание, что `created_at` помимо даты содержит и время. Если взять `created_at` как есть, то разность между датой платежа 2 ноября и созданием профиля 1 ноября в 9 утра будет 0 целых дней. Хотя более интуитивно, что разница 1 день. Поэтому для `created_at` вызовите `.dt.normalize()`, чтобы сбросить время в 0:00:00 и считать разницу именно между датами.

Идём дальше. Для каждого `passport_id` мы хотим иметь сумму платежей за первый месяц (таргет) и данные, на основе которых будем готовить признаки.

Поскольку мы хотим одну строку для каждого `passport_id`, то сгруппируем данные по `passport_id`.  А при агрегации:

- Для `type` и `created_at` возьмём `first` (тут уже без каких-то тонкостей: для каждого `passport_id` есть единственное значение, его и берём).
- Для `revenue` возьмём сумму. Это наш таргет, сумма всех платежей за первый месяц после создания профиля.

При группировке можете использовать параметр `as_index=False`, чтобы результат был в более удобном формате.

При помощи джойна (merge) добавим сюда результат прошлого степа (столбцы `user_type_name` и `user_type_cars_name`).

Теперь в датафрейме 6 столбцов. А сколько в нём строк?

In [14]:
mask_diffs = (
    pd.to_datetime(new_data["payment_date"])
    - pd.to_datetime(new_data["created_at"]).dt.normalize()
) <= timedelta(days=30)

cleared_data = (
    new_data[mask_diffs]
    .groupby(
        "passport_id",
        as_index=False,
    )
    .agg({"type": "first", "created_at": "first", "revenue": "sum"})
    .merge(passport_id_df, on="passport_id")
)
cleared_data.shape[0]

8656

### Задача 2. Обработка данных (1/5)
Есть колонки, которые всегда стоит убирать из линейных моделей.

Как думаете, есть ли среди колонок абсолютно точно лишние? 

Какую колонку вы убрали в этом кейсе?

> type

In [15]:
cleared_data.head()

,passport_id,type,created_at,revenue,user_type_name,user_type_cars_name
0,140371571,premium,2021-01-04 20:36:56,685,simple_user,None
1,140383147,premium,2021-01-06 09:01:34,685,simple_user,cars_simple
2,140386549,premium,2021-01-06 16:54:42,2055,simple_user,None
3,140387667,premium,2021-01-06 19:07:18,1370,simple_user,None
4,140390659,premium,2021-01-07 01:08:09,1370,simple_user,cars_simple


### Задача 2. Обработка данных (2/5)
Уберём выбросы.

Определите значения `2.5` и `97.5` квантилей для таргета.

Удалите все записи, значения целевой переменной которых лежат за пределами указанных квантилей. Другими словами, оставьте только те строки, у которых revenue больше или равно 2.5 квантиля и меньше или равно `97.5` квантиля.

Через запятую и пробел введите:

- Сначала значение `2.5` квантиля, округлённое до целого.
- Потом значение `97.5` квантиля, округлённое до целого.
- Потом получившееся количество строк в датафрейме.

В ответе должно получиться три целых числа, через запятую и пробел. Например, 256, 2048, 7700.

In [16]:
lower_quant = cleared_data["revenue"].quantile(0.025)
upper_quant = cleared_data["revenue"].quantile(0.975)

In [17]:
filtred_data = cleared_data[
    (cleared_data["revenue"] >= lower_quant)
    & (cleared_data["revenue"] <= upper_quant)
]

In [18]:
print(int(lower_quant), int(upper_quant), filtred_data.shape[0], sep=", ")

685, 5070, 8439


### Задача 2. Обработка данных (3/5)
Вернёмся к обсуждению пропусков в данных. В каких столбцах датафрейма есть пропуски?

In [19]:
filtred_data.isna().sum()

passport_id               0
type                      0
created_at                0
revenue                   0
user_type_name          277
user_type_cars_name    3395
dtype: int64

### Задача 2. Обработка данных (4/5)
Есть ли строки, в которых половина или более колонок пустые? Как думаете, такое возможно?

> Таких строк нет, и это невозможно, так как пропущенные значения в принципе встречаются в меньшем количестве колонок, чем половина

### Задача 2. Обработка данных (5/5)
Вспомним, как определяется `ColumnTransformer`. В списке трансформаций третьей составляющей идёт селектор. Селектор задаёт, к каким именно столбцам применяется преобразование. Например, там может быть явно задан список столбцов. Или там можно задать, что должны использоваться все числовые (или наоборот только не числовые) столбцы. Может быть и более сложная, кастомная, логика.

Продолжая нашу работу с пропусками, напишем кастомный селектор, который будет возвращать только столбцы, в которых не слишком много пропусков (не больше заданного порога). Потом его можно будет использовать в `ColumnTransformer`, автоматически отбирая только такие столбцы.

```python
class FilteringSelector:
    def __init__(self, t: float = 0.4):
        ...
        
    def __call__(self, df):
        ...
        # Возвращаем список названий "хороших" столбцов
```

Обратите внимание, что:

- Это именно селектор, не трансформер. В нём не должно быть методов fit и predict, он не должен ни от чего наследоваться.
- Возвращаем список названий столбцов, а не датафрейм.
- Возвращаем список "хороших" столбцов, в которых мало пропусков (не больше заданного порога). То есть те столбцы, которые мы хотим, чтобы были выбраны и взяты для дальнейшей работы (а не те, которые стоит отбросить).
- По умолчанию используем порог 0.4: допускаем не более 40% пропусков.
- Пример селектора был в практической части урока.

Для самопроверки попробуйте использовать написанный селектор и посмотрите, какие столбцы он вернёт.

А потом попробуйте использовать его внутри `ColumnTransformer` и убедитесь, что так тоже работает.

На проверку отправьте только сам класс `FilteringSelector`. Примеры использования отправлять не нужно (из-за особенностей проверки это может привести к ошибке и корректная реализация класса не будет принята системой).

In [20]:
class FilteringSelector:
    def __init__(self, t: float = 0.4):
        self.t = t

    def __call__(self, df):
        final_cols = df.columns[df.isna().mean() < self.t].to_list()

        # Возвращаем список названий "хороших" столбцов
        return final_cols

In [21]:
own_selector = FilteringSelector()
own_selector(filtred_data)

['passport_id', 'type', 'created_at', 'revenue', 'user_type_name']

### Задача 3. Трансформер
Теперь напишем свой sklearn-трансформер.

Обычно лучше делать трансформеры, которые решают одну конкретную задачу. Но тут мы в качестве практики напишем транформер, который делает сразу несколько вещей.

Используйте следующий шаблон кода:

```python
class AddColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, created_at_column='created_at', payments_by_month=quasi_external):
        # Ваш код здесь

    def fit(self, X, y):
        # Ваш код здесь

    def transform(self, X):
        # Ваш код здесь
```
Здесь:

- `created_at_column` — название столбца с датой создания профиля (по умолчанию `created_at`).
- `payments_by_month` — словарь с выручкой по месяцам. Сюда будет передаваться словарь `quasi_external`, который мы создали в одном из первых степов этого урока (сама переменная `quasi_external` будет доступна при проверке кода, так что не возникнет проблем с использованием её в качестве значения по умолчанию).

Трансформер делает следующие вещи:

- На основе даты создания профиля добавляет столбец `last_month_pmts` с выручкой за предыдущий месяц, используя "внешние данные" (`payments_by_month`). Если данных о платежах за предыдущий месяц нет, то берётся среднее по всем доступным месяцам (т.е. среднее во всем значениям `payments_by_month`).
- Выделяет из даты создания профиля квартал, добавляя столбец `quarter` со значением в формате '2023Q3'.
- Удаляет столбцы `passport_id` (название этой колонки можно захардкодить) и `created_at` (тут лучше взять название столбца из `created_at_column`).

Выделите из данных X и y. Проверьте на них, как работает написанный трансформер.

В поле ниже скопируйте класс `AddColumnsTransformer` и все нужные импорты (`pandas`, `BaseEstimator`, `TransformerMixin`). Пример использования отправлять на проверку не нужно. Только импорты и сам класс.

In [22]:
class AddColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self, created_at_column="created_at", payments_by_month=quasi_external
    ):
        # Ваш код здесь
        self.created_at_column = created_at_column
        self.payments_by_month = payments_by_month
        self.mean_value = sum(payments_by_month.values()) / len(
            payments_by_month
        )

    def fit(self, X, y):
        # Ваш код здесь
        return self

    def transform(self, X):
        # Ваш код здесь
        X[self.created_at_column] = pd.to_datetime(X[self.created_at_column])

        X["quarter"] = (
            pd.PeriodIndex(X[self.created_at_column], freq="Q")
            .to_series()
            .apply(lambda x: f"{x.year}Q{x.quarter}")
            .values
        )

        ext_df = pd.Series(self.payments_by_month).reset_index()
        ext_df.columns = ["payment_month", "last_month_pmts"]

        X["payment_month"] = (
            X["created_at"] - pd.DateOffset(months=1)
        ).dt.strftime("%Y-%m")

        X = pd.merge(X, ext_df, on="payment_month", how="left")

        X["last_month_pmts"].fillna(self.mean_value, inplace=True)
        X.drop(
            ["passport_id", "payment_month", self.created_at_column],
            axis=1,
            inplace=True,
        )

        return X

In [23]:
X = filtred_data.drop("revenue", axis=1)
y = filtred_data["revenue"]

In [24]:
new_tr = AddColumnsTransformer()
transformed_df = new_tr.transform(X)
transformed_df

,type,user_type_name,user_type_cars_name,quarter,last_month_pmts
0,premium,simple_user,None,2021Q1,3.114500e+06
1,premium,simple_user,cars_simple,2021Q1,3.114500e+06
2,premium,simple_user,None,2021Q1,3.114500e+06
3,premium,simple_user,None,2021Q1,3.114500e+06
4,premium,simple_user,cars_simple,2021Q1,3.114500e+06
...,...,...,...,...,...
8434,premium,simple_user,None,2023Q1,4.635815e+06
8435,premium,profi,None,2023Q1,4.635815e+06
8436,premium,simple_user,None,2023Q1,4.635815e+06
8437,premium,simple_user,cars_simple,2023Q1,4.635815e+06


### Задача 4. Пайплайн обработки данных
Соберём всю предподготовку данных вместе. Напишите пайплайн, состоящий из следующих последовательных шагов:

1. `ColumnTransformer`, использующий написанный нами селектор для автоматического отбора столбцов без большого числа пропусков. Для всех отобранных столбцов просто делаем `passthrough`.
2. `AddColumnsTransformer` с предыдущего степа. Он добавит и удалит некоторые столбцы.
3. Заполнение пропусков в данных.

Для **числовых** столбцов будем заполнять пропуски **средним** значением.

Для **категориальных** — часто заполняют модой (самым частым значением). Но мы тут поступим немного иначе. Заполним пропуски **фиксированным** значением `'unknown'`, но сделаем это с в пайплайне, с помощью `SimpleImputer` (разберитесь по документации `sklearn`, как это сделать).

Также обратите внимание, что сейчас у нас пропуски в категориальных столбцах превратились из `np.nan` в `None`. Для корректной работы `SimpleImputer` это тоже нужно учесть (разберитесь по документации `sklearn`, какой параметр отвечает за то, какое значение будет считаться пропуском; а если вдруг вы уже заменили в данных `None` на `np.nan`, то тут ничего делать не нужно).

4. Mean target encoder для категориальных столбцов. 

>**Важно**: установите параметр `shuffle=False` в `TargetEncoder` для воспроизводимости результатов.

5. `StandardScaler` для масштабирования признаков.

Чтобы лучше разобраться, можете по одному добавлять эти шаги в пайплайн и каждый раз проверять, что получается на выходе.

Проверять, как работает пайплайн, пока можно на X и y целиком. Стратегии валидации мы обсудим чуть позже.

Когда соберёте весь пайплайн, запустите его на X и y, сохраните получившийся датафрейм в csv-файл с параметром `index=False` и отправьте на проверку.

Обратите внимание, что мы запускаем пайплайн целиком на X и y только чтобы проверить, что пайплайн работает. Полученный в результате датафрейм нужен только для проверки корректности пайплайна и не будет использоваться дальше в решении. А сам пайплайн мы будем использовать немного иначе.

In [25]:
col_selector = ColumnTransformer(
    transformers=[("selector", "passthrough", FilteringSelector())],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

In [26]:
col_selector.fit_transform(filtred_data)

,passport_id,type,created_at,revenue,user_type_name
0,140371571,premium,2021-01-04 20:36:56,685,simple_user
1,140383147,premium,2021-01-06 09:01:34,685,simple_user
2,140386549,premium,2021-01-06 16:54:42,2055,simple_user
3,140387667,premium,2021-01-06 19:07:18,1370,simple_user
4,140390659,premium,2021-01-07 01:08:09,1370,simple_user
...,...,...,...,...,...
8651,144970910,premium,2023-01-28 18:48:05,1105,simple_user
8652,144972314,premium,2023-01-28 22:51:49,1100,profi
8653,144974954,premium,2023-01-29 15:12:38,785,simple_user
8654,144979202,premium,2023-01-30 10:24:40,785,simple_user


In [27]:
col_imputer = ColumnTransformer(
    transformers=[
        (
            "impute_num",
            SimpleImputer(strategy="mean"),
            selector(dtype_include="number"),
        ),
        (
            "impute_cat",
            SimpleImputer(
                strategy="constant", fill_value="unknown", missing_values=None
            ),
            selector(dtype_exclude="number"),
        ),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

In [28]:
col_transformer_with_selector = ColumnTransformer(
    transformers=[
        (
            "MeanTargetEncoder",
            TargetEncoder(target_type="continuous", shuffle=False),
            selector(dtype_exclude="number"),
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

In [29]:
pipe = Pipeline(
    [
        ("col_selector", col_selector),
        ("add_columns_transformer", AddColumnsTransformer()),
        ("col_imputer", col_imputer),
        ("col_transformer_with_selector", col_transformer_with_selector),
        ("num_scaler", StandardScaler().set_output(transform="pandas")),
    ]
)

In [30]:
X_ = filtred_data.drop(["type", "revenue"], axis=1).reset_index(drop=True)
y_ = filtred_data[["revenue"]].reset_index(drop=True)

In [31]:
final_data = pipe.fit_transform(X_, y_)
final_data.to_csv("final_data.csv", index=False)

In [32]:
final_data

,user_type_name,quarter,last_month_pmts
0,0.338846,0.147648,-0.166366
1,0.338846,0.147648,-0.166366
2,0.338846,0.147648,-0.166366
3,0.338846,0.147648,-0.166366
4,0.338846,0.147648,-0.166366
...,...,...,...
8434,-0.353409,-0.825518,0.634365
8435,1.381208,-0.825518,0.634365
8436,-0.353409,-0.825518,0.634365
8437,-0.353409,-0.825518,0.634365


### Задача 5. Валидация
В этом задании мы будем использовать `time series cross-validation`.

Не будем разбивать клиентов на обучающие и валидационные выборки случайным образом. Вместо этого будем проверять, насколько хорошо модель, обученная на "более старых" клиентах, работает для "более новых".

Для такой кросс-валидации нужны данные, упорядоченные по дате. Но вы можете убедиться, что в результате предыдущих шагов у нас уже получился датафрейм, упорядоченный по возрастанию `created_at` (проверьте, что это так; задание со звёздочкой — разобраться, почему так получилось, хотя мы специально его не сортировали, а исходные данные не были упорядочены).

Теперь создайте `TimeSeriesSplit`, установив `n_splits=4` (другие параметры задавать не нужно) и сохраните в переменную с именем `splitter`.

Отправьте на проверку эту строчку кода, добавив перед ней нужный импорт.



In [33]:
splitter = TimeSeriesSplit(n_splits=4)

### Задача 6. Обучение и выбор модели (1/5)
Обучим несколько вариаций линейной регрессии: классическую линейную регрессию, `Lasso` и `Ridge`.

Будем использовать `splitter` с предыдущего степа. А в качестве метрики — `MAE`.

Начнём с классической `LinearRegression`. Импортируйте её из `sklearn.linear_model` (можно сразу импортировать оттуда же `Lasso` и `Ridge`).

Соберите пайплайн модели, состоящий из 2 шагов: пайплайн предподготовки данных и непосредственно модель.

Проверим на кросс-валидации. Самый простой способ запустить непосредственно кросс-валидацию — использовать `cross_val_score` из `sklearn.model_selection`.

По-хорошему стоило бы выделить кусок данных в конце в качестве тестовой выборки, на которой проверим в самом конце. А кросс-валидацию делать на оставшихся данных. Со следующего урока мы так и будем делать. Но тут пока для простоты просто будем запускать кросс-валидацию на всех данных и ориентироваться на метрики кросс-валидации.

В качестве аргументов передайте итоговый пайплайн, `X`, `y`, наш сплиттер в качестве параметра `cv` и используемую метрику как параметр `scoring` (найдите, как именно нужно написать название метрики, там будет не просто `"MAE"`).

На выходе получим метрики по фолдам. Можно посчитать среднее, чтобы получить итоговое значение.

Если помимо самой кросс-валидации мы хотим заодно подобрать оптимальные гиперпараметры, то используем `GridSearchCV`.

Предварительно создадим словарь с перебираемыми гиперпараметрами. Ключи словаря состоят из названий шагов пайплайна и названия параметра, разделённых двумя подчёркиваниями. Значения — собственно, значения параметров, которые перебираем.

В качестве практики давайте попробуем значения `True` и `False` для гиперпараметра `fit_intercept` (обучаем или нет свободный коэффициент).

Создайте словарь с гиперпараметрами и экземпляр `GridSearchCV`, запустите, проверьте результат (найденные гиперпараметры и лучшее значение метрики). Проверьте, получился ли такой же результат, как при использовании `cross_val_score`, и убедитесь, что понимаете, почему.

Введите полученное значение `MAE`, округлённое до целого числа.

In [41]:
# Создаем полный пайплайн с препроцессингом и моделью
final_pipe_model = Pipeline(
    [
        ("preprocessing", pipe),  # ваш пайплайн препроцессинга
        ("model", LinearRegression()),  # модель
    ]
)

# Кросс-валидация
cv_scores = cross_val_score(
    final_pipe_model, X_, y_, cv=splitter, scoring="neg_mean_absolute_error"
).mean()
print(f"CV MAE: {-cv_scores}")

# GridSearch с правильными именами параметров
params_lr = {"model__fit_intercept": [True, False]}

grid_lr = GridSearchCV(
    final_pipe_model,  # передаем полный пайплайн
    params_lr,
    cv=splitter,
    scoring="neg_mean_absolute_error",
)
grid_lr.fit(X_, y_)
print(grid_lr.best_params_)
print(grid_lr.best_score_)
print(-round(grid_lr.best_score_))
results = pd.DataFrame(grid_lr.cv_results_)

CV MAE: 576.7235058281058
{'model__fit_intercept': True}
-576.7235058281058
577


In [42]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__fit_intercept,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
0,0.054678,0.016704,0.025298,0.001191,True,{'model__fit_intercept': True},-546.764974,-561.253953,-590.311597,-608.563500,-576.723506,24.162171,1
1,0.055310,0.015764,0.026623,0.000477,False,{'model__fit_intercept': False},-1325.257618,-1462.612498,-1344.869038,-1428.944061,-1390.420804,57.045785,2


### Задача 6. Обучение и выбор модели (2/5)
Вспомните, чем отличается Lasso-регрессия от обычной, за что отвечает параметр `alpha` и какое у него значение по умолчанию (если что-то забыли, помимо лекции может быть полезной документация sklearn).

По аналогии с предыдущим степом подготовьте пайплайн для Lasso-регрессии, состоящий из пайплайна предподготовки данных и самой модели.

Попробуем следующие значения `alpha: 0.001, 0.01, 0.1, 1, 10, 100, 1000`.

С помощью `GridSearchCV` найдите лучшее значение этого гиперпараметра и получившееся `MAE`.

Подумайте, какие выводы можно сделать из полученных результатов.

В поле ниже введите полученное значение `alpha`.

In [44]:
lasso_pipe = Pipeline([("preprocess", pipe), ("lasso", Lasso())])

params_lasso = {
    "lasso__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
}

grid_lasso = GridSearchCV(
    lasso_pipe, params_lasso, cv=splitter, scoring="neg_mean_absolute_error"
)
grid_lasso.fit(X_, y_)
grid_lasso.best_params_

{'lasso__alpha': 0.001}

In [45]:
grid_lasso.best_score_

np.float64(-576.7236023520201)

### Задача 6. Обучение и выбор модели (3/5)
Попробуем всё то же самое для `Ridge-регрессии`. Вспомните, чем она отличается от классической и от `Lasso`.

Подготовьте пайплайн, состоящий из предподготовки данных и самой модели.

Попробуем такие же значения `alpha: 0.001, 0.01, 0.1, 1, 10, 100, 1000`.

С помощью `GridSearchCV` найдите лучшее значение гиперпараметра и получившееся `MAE`.

Подумайте, какие выводы можно сделать из полученных результатов.

В поле ниже введите полученное значение `alpha`.

In [46]:
ridge_pipe = Pipeline([("preprocess", pipe), ("ridge", Ridge())])

params_ridge = {
    "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
}

grid_ridge = GridSearchCV(
    ridge_pipe, params_ridge, cv=splitter, scoring="neg_mean_absolute_error"
)
grid_ridge.fit(X_, y_)
grid_ridge.best_params_

{'ridge__alpha': 0.001}

In [50]:
grid_ridge.best_score_

np.float64(-576.7235152356632)

### Задача 6. Обучение и выбор модели (4/5)
Какая модель из трех кандидатов оказалась чуть лучше остальных?

In [56]:
print(
    max([grid_lr.best_score_, grid_lasso.best_score_, grid_ridge.best_score_])
)

-576.7235058281058
